# IBKR API notebook

#### Connection

In [1]:
from ib_async import *
util.startLoop()

ib = IB()
ib.connect('127.0.0.1', 7497, clientId=14)

<IB connected to 127.0.0.1:7497 clientId=14>

Error 321, reqId 4: Erreur de validation de la demande.-'bK' : cause - Historical data bar size setting is invalid. Legal ones are: 1 secs, 5 secs, 10 secs, 15 secs, 30 secs, 1 min, 2 mins, 3 mins, 5 mins, 10 mins, 15 mins, 20 mins, 30 mins, 1 hour, 2 hours, 3 hours, 4 hours, 8 hours, 1 day, 1W, 1M, contract: Index(symbol='NDX', exchange='NASDAQ', currency='USD')
Error 321, reqId 5: Erreur de validation de la demande.-'bK' : cause - Historical data bar size setting is invalid. Legal ones are: 1 secs, 5 secs, 10 secs, 15 secs, 30 secs, 1 min, 2 mins, 3 mins, 5 mins, 10 mins, 15 mins, 20 mins, 30 mins, 1 hour, 2 hours, 3 hours, 4 hours, 8 hours, 1 day, 1W, 1M, contract: Index(symbol='NDX', exchange='NASDAQ', currency='USD')
Peer closed connection.


## Request Historical data

#### Choose your contract

In [ ]:
contract = Stock('AAPL', 'SMART', 'USD')

In [2]:
contract = Index('NDX', 'NASDAQ', 'USD')

In [ ]:
contract = Forex('EURUSD')

#### Check first data timestamp available

In [3]:
timestamp = ib.reqHeadTimeStamp(contract, whatToShow='TRADES', useRTH=True)
formatted_time = timestamp.strftime('%B %d, %Y, %H:%M')
print(f"First date of data available: {formatted_time}")

First date of data available: March 04, 2004, 14:30


#### Request historical data function

Args
- **contract**: Contract of interest.  
- **endDateTime**:  
    - Can be set to `''` to indicate the current time.  
    - Can be given as a `datetime.date` or `datetime.datetime`.  
    - Can be given as a string in `'yyyyMMdd HH:mm:ss'` format.  
    - If no timezone is given, the TWS login timezone is used.  
- **durationStr**: Time span of all the bars. Examples:  
    - `'60 S'`, `'30 D'`, `'13 W'`, `'6 M'`, `'10 Y'`.  
- **barSizeSetting**: Time period of one bar. Must be one of:  
    - `'1 secs'`, `'5 secs'`, `'10 secs'`, `'15 secs'`, `'30 secs'`,  
    - `'1 min'`, `'2 mins'`, `'3 mins'`, `'5 mins'`, `'10 mins'`, `'15 mins'`,  
    - `'20 mins'`, `'30 mins'`,  
    - `'1 hour'`, `'2 hours'`, `'3 hours'`, `'4 hours'`, `'8 hours'`,  
    - `'1 day'`, `'1 week'`, `'1 month'`.  
- **whatToShow**: Specifies the source for constructing bars. Must be one of:  
    - `'TRADES'`, `'MIDPOINT'`, `'BID'`, `'ASK'`, `'BID_ASK'`,  
    - `'ADJUSTED_LAST'`, `'HISTORICAL_VOLATILITY'`, `'OPTION_IMPLIED_VOLATILITY'`,  
    - `'REBATE_RATE'`, `'FEE_RATE'`, `'YIELD_BID'`, `'YIELD_ASK'`, `'YIELD_BID_ASK'`, `'YIELD_LAST'`.  
    - For `'SCHEDULE'`, use `:meth:.reqHistoricalSchedule`.  
- **useRTH**:  
    - If `True`, only show data from within Regular Trading Hours.  
    - If `False`, show all data.  
- **formatDate**:  
    - For an intraday request, setting to `2` will cause the returned date fields to be timezone-aware `datetime.datetime` with UTC timezone, instead of local timezone as used by TWS.  
- **keepUpToDate**:  
    - If `True`, a realtime subscription is started to keep the bars updated.  
    - `endDateTime` must be set empty (`''`) then.  
- **chartOptions**: Unknown.  
- **timeout**:  
    - Timeout in seconds after which to cancel the request and return an empty bar series. 
    - If the data request is huge, this parameter could spoil the request  
    - Set to `0` to wait indefinitely.  


In [6]:
global historical_data_interval, duration
historical_data_interval = '1_min' 
request_duration = '10 Y'
bars = ib.reqHistoricalData(
        contract,
        endDateTime='',
        durationStr='10 Y',
        barSizeSetting='1 min',
        whatToShow='TRADES',
        useRTH=True,
        formatDate=1,
        timeout = 0)

In [7]:
bars[0]

BarData(date=datetime.datetime(2015, 4, 7, 9, 30, tzinfo=zoneinfo.ZoneInfo(key='US/Eastern')), open=4349.3, high=4354.99, low=4349.01, close=4353.94, volume=0.0, average=0.0, barCount=43)

Convert the list of bars to a data frame and print the first and last rows:

In [8]:
df = util.df(bars)

display(df.head(n=20))
display(df.tail(n=20))

,date,open,high,low,close,volume,average,barCount
0,2015-04-07 09:30:00-04:00,4349.30,4354.99,4349.01,4353.94,0.0,0.0,43
1,2015-04-07 09:31:00-04:00,4353.88,4358.52,4353.72,4357.29,0.0,0.0,58
2,2015-04-07 09:32:00-04:00,4357.31,4359.87,4356.08,4359.67,0.0,0.0,60
3,2015-04-07 09:33:00-04:00,4359.52,4361.04,4357.68,4361.04,0.0,0.0,60
4,2015-04-07 09:34:00-04:00,4361.29,4364.72,4361.29,4364.72,0.0,0.0,58
5,2015-04-07 09:35:00-04:00,4364.91,4369.10,4364.33,4368.86,0.0,0.0,59
6,2015-04-07 09:36:00-04:00,4367.84,4368.52,4365.42,4366.73,0.0,0.0,59
7,2015-04-07 09:37:00-04:00,4366.69,4368.32,4366.16,4367.34,0.0,0.0,57
8,2015-04-07 09:38:00-04:00,4367.36,4367.89,4365.30,4365.44,0.0,0.0,57
9,2015-04-07 09:39:00-04:00,4365.40,4366.18,4364.12,4365.09,0.0,0.0,59


,date,open,high,low,close,volume,average,barCount
977032,2025-04-03 12:22:00-04:00,18798.42,18813.07,18798.19,18804.53,0.0,0.0,60
977033,2025-04-03 12:23:00-04:00,18804.55,18809.48,18799.12,18804.22,0.0,0.0,59
977034,2025-04-03 12:24:00-04:00,18801.67,18804.02,18795.40,18803.73,0.0,0.0,60
977035,2025-04-03 12:25:00-04:00,18804.32,18805.07,18771.62,18774.56,0.0,0.0,60
977036,2025-04-03 12:26:00-04:00,18772.53,18772.54,18761.18,18762.47,0.0,0.0,60
977037,2025-04-03 12:27:00-04:00,18762.70,18773.79,18759.17,18773.79,0.0,0.0,60
977038,2025-04-03 12:28:00-04:00,18774.05,18787.87,18774.05,18787.87,0.0,0.0,59
977039,2025-04-03 12:29:00-04:00,18788.70,18802.91,18788.70,18802.41,0.0,0.0,60
977040,2025-04-03 12:30:00-04:00,18804.16,18817.43,18783.31,18783.31,0.0,0.0,60
977041,2025-04-03 12:31:00-04:00,18783.58,18783.59,18766.86,18768.22,0.0,0.0,60


Save your pulled data in a dataframe

Compression possibilities sorted by compression ratio from the lowest to the highest : 
- `snappy`

- `gzip`

- `brotli`

#### Checking if volume and average columns are empty or not and remove it if empty

In [9]:
# Check if the 'volume' column is empty (all values are 0.0)
if (df['volume'] == 0.0).all():
    df = df.drop(columns=['volume'])  # Drop the 'volume' column
    print("The 'volume' column was empty and has been removed.")

# Check if the 'average' column is empty (all values are 0.0)
if (df['average'] == 0.0).all():
    df = df.drop(columns=['average'])  # Drop the 'average' column
    print("The 'average' column was empty and has been removed.")

# Display the updated DataFrame
display(df.head(n=30))

The 'volume' column was empty and has been removed.
The 'average' column was empty and has been removed.


,date,open,high,low,close,barCount
0,2015-04-07 09:30:00-04:00,4349.30,4354.99,4349.01,4353.94,43
1,2015-04-07 09:31:00-04:00,4353.88,4358.52,4353.72,4357.29,58
2,2015-04-07 09:32:00-04:00,4357.31,4359.87,4356.08,4359.67,60
3,2015-04-07 09:33:00-04:00,4359.52,4361.04,4357.68,4361.04,60
4,2015-04-07 09:34:00-04:00,4361.29,4364.72,4361.29,4364.72,58
5,2015-04-07 09:35:00-04:00,4364.91,4369.10,4364.33,4368.86,59
6,2015-04-07 09:36:00-04:00,4367.84,4368.52,4365.42,4366.73,59
7,2015-04-07 09:37:00-04:00,4366.69,4368.32,4366.16,4367.34,57
8,2015-04-07 09:38:00-04:00,4367.36,4367.89,4365.30,4365.44,57
9,2015-04-07 09:39:00-04:00,4365.40,4366.18,4364.12,4365.09,59


In [11]:
# Récupérer la devise et l'unité de temps
symbol = contract.symbol

# Construire le nom du fichier
save_path = f"../database/{symbol}_{historical_data_interval}.parquet"

# Sauvegarder le DataFrame en fichier parquet
df.to_parquet(save_path, index=True, compression=None)
print(f"Fichier sauvegardé sous le nom : {save_path}")

Fichier sauvegardé sous le nom : ../database/NDX_1_min.parquet


Instruct the notebook to draw plot graphics inline:

In [ ]:
%matplotlib inline

Plot the close data

In [ ]:
df.plot(y='close');

There is also a utility function to plot bars as a candlestick plot. It can accept either a DataFrame or a list of bars. Here it will print the last 100 bars:

In [ ]:
util.barplot(bars[-100:], title=contract.symbol);

## Historical data with realtime updates

A new feature of the API is to get live updates for historical bars. This is done by setting `endDateTime` to an empty string and the `keepUpToDate` parameter to `True`.

Let's get some bars with an keepUpToDate subscription:

In [ ]:

bars = ib.reqHistoricalData(
        contract,
        endDateTime='',
        durationStr='900 S',
        barSizeSetting='10 secs',
        whatToShow='MIDPOINT',
        useRTH=True,
        formatDate=1,
        keepUpToDate=True)

Replot for every change of the last bar:

In [ ]:
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

def onBarUpdate(bars, hasNewBar):
    plt.close()
    plot = util.barplot(bars)
    clear_output(wait=True)
    display(plot)

bars.updateEvent += onBarUpdate

ib.sleep(10)
ib.cancelHistoricalData(bars)

Realtime bars
------------------

With ``reqRealTimeBars`` a subscription is started that sends a new bar every 5 seconds.

First we'll set up a event handler for bar updates:

In [ ]:
def onBarUpdate(bars, hasNewBar):
    print(bars[-1])

Then do the real request and connect the event handler,

In [ ]:
bars = ib.reqRealTimeBars(contract, 5, 'MIDPOINT', False)
bars.updateEvent += onBarUpdate

let it run for half a minute and then cancel the realtime bars.

In [ ]:
ib.sleep(30)
ib.cancelRealTimeBars(bars)

The advantage of reqRealTimeBars is that it behaves more robust when the connection to the IB server farms is interrupted. After the connection is restored, the bars from during the network outage will be backfilled and the live bars will resume.

reqHistoricalData + keepUpToDate will, at the moment of writing, leave the whole API inoperable after a network interruption.

### Request historical market news

In [ ]:
ib.reqHistoricalNews()
#ib.reqHistoricalNewsAsync()

In [ ]:
ib.disconnect()